# PA3

In [ ]:
# %pip install databricks-vectorsearch
# %pip install langchain_community
# %pip install langgraph
# dbutils.library.restartPython()


### Task 0: Infrastructure Provisioning

In [14]:
import os
from dotenv import load_dotenv
from pathlib import Path 

import re
import time

import numpy as np
import pandas as pd
from typing import Iterator

from openai import AzureOpenAI
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import (
    ChatPromptTemplate, 
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)

from databricks.vector_search.client import VectorSearchClient

from pyspark.sql.functions import udf, explode, col, expr, pandas_udf
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, FloatType

load_dotenv()

# THE ABOVE IS WHAT YOU NEED FOR THIS ASSIGNMENT

True

In [2]:
curr_dir = Path.cwd()
target_env_path = curr_dir / ".env"
success = load_dotenv(target_env_path, override=True)

print("AZURE_OPENAI_API_VERSION:", "SET" if os.getenv("AZURE_OPENAI_API_VERSION") else "NOT SET")
print("AZURE_OPENAI_API_KEY:", "SET" if os.getenv("AZURE_OPENAI_API_KEY") else "NOT SET")
print("AZURE_OPENAI_ENDPOINT:", "SET" if os.getenv("AZURE_OPENAI_ENDPOINT") else "NOT SET")
print("DATABRICKS_HOST:", "SET" if os.getenv("DATABRICKS_HOST") else "NOT SET")
print("DATABRICKS_TOKEN:", "SET" if os.getenv("DATABRICKS_TOKEN") else "NOT SET")

AZURE_OPENAI_API_VERSION: SET
AZURE_OPENAI_API_KEY: SET
AZURE_OPENAI_ENDPOINT: SET
DATABRICKS_HOST: SET
DATABRICKS_TOKEN: SET


In [15]:
from databricks.connect import DatabricksSession

# Ensure these are actually set in your environment
spark = DatabricksSession.builder \
    .host(os.getenv("DATABRICKS_HOST")) \
    .token(os.getenv("DATABRICKS_TOKEN")) \
    .serverless() \
    .getOrCreate()

# Test it
spark.sql("SELECT 2").show()

+---+
|  2|
+---+
|  2|
+---+



In [6]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient(
    host=os.getenv("DATABRICKS_HOST"),
    token=os.getenv("DATABRICKS_TOKEN"),
    auth_type='pat' 
)

dbutils = w.dbutils

## Part 0: Infrastructure Provisioning

In [22]:
spark.sql(
    """
    CREATE CATALOG IF NOT EXISTS `24280021-pa3`
    """
)

""


In [33]:
catalog = "`24280021-pa3`"
volumes = ["rag_documents", "text_documents"]

for volume in volumes:
    query = f"CREATE VOLUME IF NOT EXISTS {catalog}.default.{volume}"
    print(f"Running: {query}")
    spark.sql(query)

print("All volumes created successfully!")

Running: CREATE VOLUME IF NOT EXISTS `24280021-pa3`.default.rag_documents
Running: CREATE VOLUME IF NOT EXISTS `24280021-pa3`.default.text_documents
All volumes created successfully!


In [29]:
catalog = "`24280021-pa3`"
schemas = ["parsed_data", "knowledge_base_data", "processed_data", "vector_index"]

for schema in schemas:
    query = f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}"
    print(f"Running: {query}")
    spark.sql(query)

print("All schemas created successfully!")

Running: CREATE SCHEMA IF NOT EXISTS `24280021-pa3`.parsed_data
Running: CREATE SCHEMA IF NOT EXISTS `24280021-pa3`.knowledge_base_data
Running: CREATE SCHEMA IF NOT EXISTS `24280021-pa3`.processed_data
Running: CREATE SCHEMA IF NOT EXISTS `24280021-pa3`.vector_index
All schemas created successfully!


## Part 1: RAG Implementation from Scratch

### Task 1: Constructing the Knowledge Base

#### Task 1.1

In [4]:
rag_path = "/Volumes/24280021-pa3/default/rag_documents"
txt_path = "/Volumes/24280021-pa3/default/text_documents"

In [ ]:
rag_doc = dbutils.fs.ls(rag_path)[0][0]

'/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf'

##### Read pdf directory

In [ ]:
df = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.pdf") \
    .load(rag_path)

df.select("path", "modificationTime", "length", "content").show()

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|dbfs:/Volumes/242...|2026-04-19 14:30:09|802199|[25 50 44 46 2D 3...|
+--------------------+-------------------+------+--------------------+



In [ ]:
parsed_df = df.select(
    col("path").alias("document_path"),
    expr("ai_parse_document(content)").alias("parsed_document")
)

display(parsed_df)

DataFrame[document_path: string, parsed_document: variant]

In [7]:
parsed_df = df.select(
    col("path").alias("document_path"),
    # 1. ai_parse_document creates the complex JSON structure
    # 2. We extract the 'elements' array
    # 3. We transform it by pulling only the 'content' string from each element
    # 4. We join them all together with double line breaks
    expr("""
        concat_ws('\\n\\n', transform(
                try_cast(
                    ai_parse_document(content):document:elements AS ARRAY<VARIANT>
                ), 
                e -> try_cast(e:content AS STRING)
            )
        )
    """).alias("parsed_document")
)

In [19]:
parsed_df.show(truncate=False)

+-----------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##### Read txt directory

In [ ]:
df2 = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.txt") \
    .load(txt_path)

df2.select("path", "modificationTime", "length", "content").show()

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|dbfs:/Volumes/242...|2026-04-19 14:30:50|  1927|[54 68 65 20 44 6...|
|dbfs:/Volumes/242...|2026-04-19 14:30:50|  1904|[41 7A 75 72 65 2...|
+--------------------+-------------------+------+--------------------+



In [15]:
parsed_df2 = df2.select(
    col("path").alias("document_path"),
    col("content").cast("string").alias("parsed_document")
)

parsed_df2.show(truncate=False)

+---------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##### Combine pdf + txt files in a df

In [ ]:
combined_df = parsed_df.unionByName(parsed_df2)

combined_df.show(truncate=False)

+-----------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##### Write combined_pdf to a delta table

In [21]:
(combined_df.write 
    .format("delta") 
    .mode("append") 
    .saveAsTable("`24280021-pa3`.parsed_data.parsed_documents")
)
print("Successfully saved to parsed_data.parsed_documents")

Successfully saved to parsed_data.parsed_documents


In [23]:
spark.sql(f"SELECT * FROM `24280021-pa3`.parsed_data.parsed_documents").show()

+--------------------+--------------------+
|       document_path|     parsed_document|
+--------------------+--------------------+
|dbfs:/Volumes/242...|Demystifying and ...|
|dbfs:/Volumes/242...|Azure Cloud Ecosy...|
|dbfs:/Volumes/242...|The Databricks Da...|
+--------------------+--------------------+



#### Task 1.2

##### Creating the table before extracting the data

In [32]:
spark.sql(
    """ 
    CREATE TABLE IF NOT EXISTS `24280021-pa3`.knowledge_base_data.knowledge_base (
        Id BIGINT GENERATED ALWAYS AS IDENTITY,
        Title STRING,
        Authors STRING,
        Content STRING,
        DocumentURI STRING
        )
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true'
    )
    """
)

print("knowledge_base_data.knowledge_base table created with CDF enabled")

knowledge_base_data.knowledge_base table created with CDF enabled


##### Extracting the data

In [4]:
combined_df = spark.read.table("`24280021-pa3`.parsed_data.parsed_documents")
combined_df

DataFrame[document_path: string, parsed_document: string]

In [5]:
extracted_df = combined_df.select(
    col("document_path").alias("DocumentURI"),
    expr("ai_extract(parsed_document, array('Title', 'Authors'))").alias("ai_data"),
    col("parsed_document").alias("Content")
)
extracted_df.show(truncate=False)

+-----------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
extracted_df = combined_df.select(
    col("document_path").alias("DocumentURI"),
    expr("ai_extract(parsed_document, array('Title', 'Authors', 'Content'))").alias("ai_data")
)
extracted_df.show(truncate=False)

+-----------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
# Extracting the required columns from the nested ai_data struct
final_extracted_df = extracted_df.select(
    col("ai_data.Title").alias("Title"),
    col("ai_data.Authors").alias("Authors"),
    col("Content"),
    col("DocumentURI")
)

final_extracted_df.show(truncate=False)

+-------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

##### Writing the extracted data to the table

In [10]:
(final_extracted_df.write
 .mode("append")
 .saveAsTable("`24280021-pa3`.knowledge_base_data.knowledge_base")
)

In [11]:
spark.sql("SELECT * FROM `24280021-pa3`.knowledge_base_data.knowledge_base").show()

+---+--------------------+--------------------+--------------------+--------------------+
| Id|               Title|             Authors|             Content|         DocumentURI|
+---+--------------------+--------------------+--------------------+--------------------+
|  1|Demystifying and ...|["Tiannuo Yang","...|Demystifying and ...|dbfs:/Volumes/242...|
|  2|Azure Cloud Ecosy...|                NULL|Azure Cloud Ecosy...|dbfs:/Volumes/242...|
|  3|The Databricks Da...|                NULL|The Databricks Da...|dbfs:/Volumes/242...|
+---+--------------------+--------------------+--------------------+--------------------+



### Task 2: Implement Chunking Strategies

##### Chunking Implementation

In [13]:
class MyTextSplitter:
    def __init__(self, chunk_size=512, chunk_overlap=0):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def split_text(self, text):            
        chunks = []
        start = 0
        
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            chunks.append(text[start:end])
            
            # Stepping forward by chunk_size minus the overlap to create the sliding window
            start += self.chunk_size - self.chunk_overlap
            
        return chunks

In [20]:
db_text = spark.sql("SELECT Content FROM `24280021-pa3`.knowledge_base_data.knowledge_base").collect()
db_text

[Row(Content='Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents\n\nTiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,\n\nGang Wang1, Xiaoguang Liu1\n\n1 Nankai University\n\n2 University of Illinois Urbana-Champaign\n\n{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn\n\nbowenj4@illinois.edu\n\nAbstract\n\nLarge Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation. Second,\nwe identify inefficiencies in system design, including improper scheduling an

In [60]:
text_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)

all_chunks = []
for row in db_text:
    row_chunks = text_splitter.split_text(row.Content)
    all_chunks.extend(row_chunks)
len(all_chunks)

149

In [61]:
all_chunks[:5]

['Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents\n\nTiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,\n\nGang Wang1, Xiaoguang Liu1\n\n1 Nankai University\n\n2 University of Illinois Urbana-Champaign\n\n{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn\n\nbowenj4@illinois.edu\n\nAbstract\n\nLarge Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through i',
 'decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation. Second,\nwe identify inefficiencies in sys

In [14]:
class MySemanticSplitter:
    def __init__(self, client, model_name, distance_threshold=0.3, buffer_size=1):
        self.client = client 
        self.model_name = model_name
        self.distance_threshold = distance_threshold
        self.buffer_size = buffer_size
    

    def _combine_sentences(self, sentences, buffer_size):
        """
        Combines each sentence with its surrounding sentences based on buffer_size.
        Returning a list of dictionaries so we don't lose the original sentence.
        """
        combined = []
        for i in range(len(sentences)):
            # Create a sliding window (e.g., if buffer is 1, window is size 3)
            start_index = max(0, i - buffer_size)
            end_index = min(len(sentences), i + 1 + buffer_size)
            
            # Joining the window into a single padded string
            window = sentences[start_index:end_index]
            combined_sentence = " ".join(window)
            
            # Keeping both original and padded versions
            combined.append({
                "sentence": sentences[i], 
                "combined_sentence": combined_sentence
            })
        return combined


    @staticmethod
    def cosine_similarity(embeddings1, embeddings2):
        vec1, vec2 = np.array(embeddings1), np.array(embeddings2)
        dot_product = np.dot(vec1, vec2)
        norm1, norm2 = np.linalg.norm(vec1), np.linalg.norm(vec2)
        
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return dot_product / (norm1 * norm2)


    def _get_embedding(self, text):
        # Using the standard OpenAI/Databricks format.
        response = self.client.embeddings.create(
            model=self.model_name,
            input=text
        )
        return response.data[0].embedding


    def split_text(self, text):
        if not text:
            return []

        # Splitting sentences on punctuation followed by a space (. ? !)
        raw_sentences = re.split(r'(?<=[.?!])\s+', text)
        sentences = [s.strip() for s in raw_sentences if s.strip()]
        
        if len(sentences) <= 1:
            return sentences

        # Combining each sentence with adjacent buffer sentences
        combined_dicts = self._combine_sentences(sentences, self.buffer_size)
        
        # Vector embedding of each padded sentence
        embeddings = [self._get_embedding(item["combined_sentence"]) for item in combined_dicts]
        
        distances = []
        for i in range(len(embeddings) - 1):
            sim = self.cosine_similarity(embeddings[i], embeddings[i+1])
            distances.append(1.0 - sim)
            
        chunks = []
        current_chunk = [sentences[0]] 
        
        for i, distance in enumerate(distances):
            if distance > self.distance_threshold:
                # Semantic shift detected, so starting a new chunk.
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i+1]]
            else:
                current_chunk.append(sentences[i+1])
        
        # Final chunk
        if current_chunk:
            chunks.append(" ".join(current_chunk))
            
        return chunks

##### SANITY CHECK

In [ ]:
# Sample Research Paper
sample_document = (
    "Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and discuss their roles in creating secure, scalable, and cost-effective data engineering workflows. We also present an experiment that is designed to bridge the gap between theory and practice by demonstrating the ingestion of data, feature engineering, and basic fraud detection. We utilized a synthetic dataset of financial transactions. The experimental procedure and metrics such as query latency, processing throughput, speed, and performance results are described in detail offering a reproducible benchmark for evaluating similar workloads in Databricks. This work is designed to function as a technical resource for individuals in industry, data engineers, and researchers who are interested in working with the Databricks environment for large-scale analytics. Keywords: Databricks, Lakehouse, Delta Lake, Data Engineering, PySpark, Unity Catalog, MLflow, Azure Synapse, Data Factory, Power BI, Real-time Analytics, Cloud Data Platforms, Fraud Detection, Performance Benchmarking, Industry Use Case. 1. Introduction These days organizations have high volume of data in the form of structured, semistructured and unstructured data. They need platforms that can integrate storage, analytics and machine learning at scale in order to gain value from their data. Conventional data warehouses store structured data for analytical needs (such as reporting through Power BI), but lack flexibility. Data lakes, on the other hand, can store semistructured and unstructured data using batch data processing, but do not provide ACID guarantees. The new Lakehouse model provides the best of both worlds to store and..."
)

# --- Test 1: The Normal Text Splitter (Fixed Size) ---
print("====== FIXED-SIZE CHUNKER ======")
normal_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)
normal_chunks = normal_splitter.split_text(sample_document)

for i, chunk in enumerate(normal_chunks[:3]):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)


# --- Test 2: The Semantic Splitter (Embedding-Based) ---
print("\n\n====== SEMANTIC CHUNKER ======")

azure_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)


semantic_splitter = MySemanticSplitter(
    client = azure_client,
  #  client=get_client(), 
    model_name=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"), 
    distance_threshold=0.15, 
    buffer_size=1
)
semantic_chunks = semantic_splitter.split_text(sample_document)

for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)

====== FIXED-SIZE CHUNKER ======

--- Chunk 1 (Length: 512) ---
Abstract: The exponential growth in data has created a need for platforms capable of storing both structured & unstructured data, effectively processing the data, analyzing, and creating machine learning models. Traditional data lakes and warehouses often lack the flexibility and performance to provide these capabilities. So, the new Lakehouse paradigm is introduced. Databricks is an implementation of the Lakehouse paradigm that is cloud-native and built upon Apache Spark. However, there is a lack of substa

--- Chunk 2 (Length: 512) ---
n Apache Spark. However, there is a lack of substantial academic work describing the ecosystem. This paper presents a comprehensive description of the Databricks ecosystem, showing it both as an architecture and as a platform already in use. We will go through the main components of Databricks architecture, including Delta Lake, Unity Catalog, cluster management, Azure integrations, and di

In [21]:
semantic_chunks2 = semantic_splitter.split_text(db_text[0].Content)

for i, chunk in enumerate(semantic_chunks2):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)


--- Chunk 1 (Length: 631) ---
Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents

Tiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,

Gang Wang1, Xiaoguang Liu1

1 Nankai University

2 University of Illinois Urbana-Champaign

{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn

bowenj4@illinois.edu

Abstract

Large Language Model (LLM)-based search agents have shown remarkable ca-
pabilities in solving complex tasks by dynamically decomposing problems and
addressing them through interleaved reasoning and retrieval. However, this in-
terleaved paradigm introduces substantial efficiency bottlenecks.

--- Chunk 2 (Length: 245) ---
First, we ob-
serve that both highly accurate and overly approximate retrieval methods de-
grade system efficiency: exact search incurs significant retrieval overhead, while
coarse retrieval requires additional reasoning steps during generation.

--- Chunk 3 (Length: 757) ---
Second,
we identify inefficie

##### Writing fixed_chunks in in processed_data

In [ ]:
text_splitter = MyTextSplitter(chunk_size=512, chunk_overlap=50)

# 1. Load the knowledge base table
db_text = spark.sql("SELECT * FROM `24280021-pa3`.knowledge_base_data.knowledge_base").collect()

# 2. Split the content into chunks
all_chunks = []
for row in db_text:
    chunks = []
    row_chunks = text_splitter.split_text(row.Content)
    print(f"\nDocument URI: {row.DocumentURI}")
    for i, chunk in enumerate(row_chunks):
        chunks.append([row.DocumentURI, i, chunk])
    all_chunks.extend(chunks)

    for i, chunk in enumerate(row_chunks[:3]):  # Print only the first 3 chunks for brevity
        print(f"Chunk {i+1} (Length: {len(chunk)}): {chunk[:100]}...")  # Print first 100 chars of each chunk

print(f"\nTotal Chunks Created: {len(all_chunks)}")

# 3. Write the chunks to a new table



Document URI: dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf
Chunk 1 (Length: 512): Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents

Tiannuo Yang1...
Chunk 2 (Length: 512): decomposing problems and
addressing them through interleaved reasoning and retrieval. However, this ...
Chunk 3 (Length: 512): esign, including improper scheduling and
frequent retrieval stalls, which lead to cascading latency—...

Document URI: dbfs:/Volumes/24280021-pa3/default/text_documents/azure_info.txt
Chunk 1 (Length: 512): Azure Cloud Ecosystem Overview
Microsoft Azure is a comprehensive cloud computing platform providin...
Chunk 2 (Length: 512): ified.

Core Infrastructure and Storage
The backbone of Azure data storage is Azure Data Lake Sto...
Chunk 3 (Length: 512): lution" and connectivity issues often found in secure environments.

Data and AI Services
Azure p...

Document URI: dbfs:/Volumes/24280021-pa3/default/text_documents/databricks_in

In [ ]:
df = spark.createDataFrame(all_chunks, schema=["DocumentURI", "ChunkID", "ChunkContent"])

display(df.limit(5))

,DocumentURI,ChunkID,ChunkContent
0,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,0,"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents\n\nTiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,\n\nGang Wang1, Xiaoguang Liu1\n\n1 Nankai University\n\n2 University of Illinois Urbana-Champaign\n\n{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn\n\nbowenj4@illinois.edu\n\nAbstract\n\nLarge Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through i"
1,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,1,"decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks. First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation. Second,\nwe identify inefficiencies in system design, including improper scheduling and\nfrequent"
2,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,2,"esign, including improper scheduling and\nfrequent retrieval stalls, which lead to cascading latency—where even minor de-\nlays in retrieval amplify end-to-end inference time. To address these challenges,\nwe introduce SearchAgent-X, a high-efficiency inference framework for LLM-\nbased search agents. SearchAgent-X leverages high-recall approximate retrieval\nand incorporates two key techniques: priority-aware scheduling and non-stall\nretrieval. Extensive experiments demonstrate that SearchAgent-X consistently\no"
3,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,3,"ents demonstrate that SearchAgent-X consistently\noutperforms state-of-the-art systems such as vLLM and HNSW-based retrieval\nacross diverse tasks, achieving up to 3.4× higher throughput and 5× lower la-\ntency, without compromising generation quality. SearchAgent-X is available at\nhttps://github.com/tiannuo-yang/SearchAgent-X.\n\n1 Introduction\n\nTraditional Retrieval-Augmented Generation (RAG) typically uses a sequential retrieve-then-generate\napproach [1–8], which limits dynamic interaction with knowledge base"
4,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,4,"ich limits dynamic interaction with knowledge bases. Recent advancements\nhave ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong\nreasoning capabilities of Large Language Models (LLMs), allowing for the dynamic and adaptive\ninterleaving of reasoning steps with retrieval calls throughout the generation process. Instead of a\nfixed pipeline, search agents can decide when and what to retrieve based on LLM’s ongoing reasoning,\nleading to significant improvements in the quality a"


In [46]:
(df.write.format("delta")
        .mode("append")
        .saveAsTable("`24280021-pa3`.processed_data.fixed_chunks")
)
print("Successfully saved fixed chunks to processed_data.fixed_chunks")

Successfully saved fixed chunks to processed_data.fixed_chunks


##### Writing semantic_chunks in in processed_data

In [47]:
client = AzureOpenAI(
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
)

semantic_splitter = MySemanticSplitter(
    client=client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    distance_threshold=0.15
)

# 1. Load the knowledge base table
db_text = spark.sql("SELECT * FROM `24280021-pa3`.knowledge_base_data.knowledge_base").collect()

# 2. Split the content into chunks
all_semantic_chunks = []
for row in db_text:
    chunks = []
    row_chunks = semantic_splitter.split_text(row.Content)
    print(f"\nDocument URI: {row.DocumentURI}")
    for i, chunk in enumerate(row_chunks):
        chunks.append([row.DocumentURI, i, chunk])
    all_semantic_chunks.extend(chunks)

    for i, chunk in enumerate(row_chunks[:3]):  # Example chunks
        print(f"Chunk {i+1} (Length: {len(chunk)}): {chunk[:100]}...")  

print(f"\nTotal Semantic Chunks Created: {len(all_semantic_chunks)}")

# 3. Write the chunks to a new table



Document URI: dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf
Chunk 1 (Length: 631): Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents

Tiannuo Yang1...
Chunk 2 (Length: 245): First, we ob-
serve that both highly accurate and overly approximate retrieval methods de-
grade sys...
Chunk 3 (Length: 757): Second,
we identify inefficiencies in system design, including improper scheduling and
frequent retr...

Document URI: dbfs:/Volumes/24280021-pa3/default/text_documents/azure_info.txt
Chunk 1 (Length: 250): Azure Cloud Ecosystem Overview
Microsoft Azure is a comprehensive cloud computing platform providin...
Chunk 2 (Length: 477): For data engineering and AI, the platform's primary strength lies in its "Enterprise Integration," w...
Chunk 3 (Length: 259): This is often paired with Azure Networking, where Virtual Networks (VNets) and Private Endpoints ens...

Document URI: dbfs:/Volumes/24280021-pa3/default/text_documents/databri

In [48]:
df2 = spark.createDataFrame(all_semantic_chunks, schema=["DocumentURI", "ChunkID", "ChunkContent"])

display(df2.limit(5))

,DocumentURI,ChunkID,ChunkContent
0,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,0,"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents\n\nTiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,\n\nGang Wang1, Xiaoguang Liu1\n\n1 Nankai University\n\n2 University of Illinois Urbana-Champaign\n\n{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn\n\nbowenj4@illinois.edu\n\nAbstract\n\nLarge Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks."
1,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,1,"First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation."
2,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,2,"Second,\nwe identify inefficiencies in system design, including improper scheduling and\nfrequent retrieval stalls, which lead to cascading latency—where even minor de-\nlays in retrieval amplify end-to-end inference time. To address these challenges,\nwe introduce SearchAgent-X, a high-efficiency inference framework for LLM-\nbased search agents. SearchAgent-X leverages high-recall approximate retrieval\nand incorporates two key techniques: priority-aware scheduling and non-stall\nretrieval. Extensive experiments demonstrate that SearchAgent-X consistently\noutperforms state-of-the-art systems such as vLLM and HNSW-based retrieval\nacross diverse tasks, achieving up to 3.4× higher throughput and 5× lower la-\ntency, without compromising generation quality."
3,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,3,"SearchAgent-X is available at\nhttps://github.com/tiannuo-yang/SearchAgent-X. 1 Introduction\n\nTraditional Retrieval-Augmented Generation (RAG) typically uses a sequential retrieve-then-generate\napproach [1–8], which limits dynamic interaction with knowledge bases."
4,dbfs:/Volumes/24280021-pa3/default/rag_documents/rag_research_paper.pdf,4,"Recent advancements\nhave ushered in RAG 2.0, known as Search Agents [9–16]. This paradigm leverages the strong\nreasoning capabilities of Large Language Models (LLMs), allowing for the dynamic and adaptive\ninterleaving of reasoning steps with retrieval calls throughout the generation process."


In [49]:
(df2.write.format("delta")
        .mode("append")
        .saveAsTable("`24280021-pa3`.processed_data.semantic_chunks")
)
print("Successfully saved semantic chunks to processed_data.semantic_chunks")

Successfully saved semantic chunks to processed_data.semantic_chunks


### Task 3: Generating Vector Embeddings

In [19]:
azure_client = AzureOpenAI(
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
)

class GenerateEmbeddings:
    def __init__(self, client, model_name, chunk_table_name):
        self.client = client
        self.model_name = model_name
        self.chunk_table_name = chunk_table_name

    def _get_embedding(self, text):
        response = self.client.embeddings.create(
            model=self.model_name,
            input=text
        )
        return response.data[0].embedding

    def embed(self):
        records = []
        chunk_df = spark.read.table(self.chunk_table_name) 
        chunks = chunk_df.collect() 
        
        print(f"Generating embeddings for {len(chunks)} chunks... This might take a minute.")

        for chunk in chunks:
            text = chunk['ChunkContent']
            embedding = self._get_embedding(text)
            records.append((text, embedding))
            
        self.embeddings = spark.createDataFrame(records, ["ChunkText", "Embedding"])
        print("Embeddings generated successfully!")

    def save_to_table(self, table_name):
        self.embeddings.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Saved to {table_name}")

In [0]:
azure_client = AzureOpenAI(
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
)

class GenerateEmbeddings:
    def __init__(self, client, model_name, chunk_table_name):
        self.client = client
        self.model_name = model_name
        self.chunk_table_name = chunk_table_name

    def embed(self):
        records = []
        chunk_df = spark.read.table(self.chunk_table_name) 
        chunks = chunk_df.collect()
        for i, chunk in enumerate(chunks):
            response = self.client.embeddings.create(
                model=self.model_name,
                input=chunk['ChunkContent']
            )
            records.append((chunk['ChunkContent'], response.data[0].embedding))
        self.embeddings = spark.createDataFrame(records, ["ChunkText", "Embedding"])

    def save_to_table(self, table_name):
        self.embeddings.write.format("delta").mode("overwrite").saveAsTable(table_name)


In [20]:
fixed_embeddings = GenerateEmbeddings(
    client=azure_client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name="`24280021-pa3`.processed_data.fixed_chunks"
)

table_name = "`24280021-pa3`.processed_data.fixed_embeddings"
fixed_embeddings.embed()
fixed_embeddings.save_to_table(table_name)

Generating embeddings for 149 chunks... This might take a minute.
Embeddings generated successfully!
Saved to `24280021-pa3`.processed_data.fixed_embeddings


In [31]:
chunk_table_name = "`24280021-pa3`.processed_data.semantic_embeddings"
chunks = spark.read.table(chunk_table_name) 

In [32]:
chunks

,ChunkText,Embedding
0,"Demystifying and Enhancing the Efficiency of Large Language Model Based Search Agents\n\nTiannuo Yang1*, Zebin Yao1*, Bowen Jin2, Lixiao Cui1, Yusen Li1,\n\nGang Wang1, Xiaoguang Liu1\n\n1 Nankai University\n\n2 University of Illinois Urbana-Champaign\n\n{yangtn,yaozb,cuilx,liyusen,wgzwp,liuxg}@nbjl.nankai.edu.cn\n\nbowenj4@illinois.edu\n\nAbstract\n\nLarge Language Model (LLM)-based search agents have shown remarkable ca-\npabilities in solving complex tasks by dynamically decomposing problems and\naddressing them through interleaved reasoning and retrieval. However, this in-\nterleaved paradigm introduces substantial efficiency bottlenecks.","[0.016945188865065575, 0.0072133201174438, 0.06323269754648209, 0.016064919531345367, -0.01596711203455925, 0.03831617906689644, -0.017079675570130348, 0.03772933408617973, -0.02728835679590702, 0.034990716725587845, 0.02963574230670929, -0.07443168014287949, -0.05531027168035507, 0.013228495605289936, -0.0027187492232769728, -0.017079675570130348, 0.012079254724085331, -0.02419518679380417, 0.03337688744068146, 0.03036930039525032, 0.062107909470796585, -0.028657665476202965, 0.006250525359064341, 0.02369392290711403, 0.020319556817412376, -0.00484453933313489, 0.013424111530184746, 0.03513742610812187, 0.013913149945437908, -0.029268963262438774, 0.024121832102537155, -0.02582124061882496, -0.015991564840078354, 0.018534565344452858, -0.03486845642328262, 0.06284146755933762, 0.007347805891185999, 0.001728445990011096, -0.006369728595018387, 0.009028876200318336, 0.00835033506155014, 0.010031405836343765, -0.006754846312105656, -0.009615723043680191, 0.01635834388434887, 0.04161718860268593, 0.0062138475477695465, 0.01201812457293272, -0.016957415267825127, 0.003310180502012372, 0.01798439584672451, 0.001749841496348381, 0.0036708463449031115, -0.05804888904094696, -0.012702778913080692, -0.03550420701503754, 0.013350754976272583, -0.010789415799081326, 0.016774026677012444, 0.024904293939471245, -0.036677900701761246, -0.027019385248422623, 0.0008489405154250562, 0.048610441386699677, -0.04587182775139809, -0.04467368125915527, -0.028339790180325508, -0.010410410352051258, -0.04381786286830902, 0.0026362240314483643, -0.024024024605751038, -0.007476178463548422, 0.0672428160905838, -0.003377423156052828, 0.010184230282902718, -0.009725756011903286, -0.009303960017859936, 0.06787856668233871, -0.015673689544200897, 0.012898394837975502, 0.012715005315840244, 0.004364669788628817, 0.02890218421816826, -0.05741313844919205, -0.026359183713793755, -0.006106870248913765, 0.00484453933313489, 0.019341478124260902, -0.03176306188106537, -0.05501684918999672, -0.028877733275294304, 0.036506734788417816, 0.0018873835215345025, -0.009114458225667477, 0.04183725640177727, 0.03772933408617973, -0.006122152786701918, 0.009878580458462238, 0.0739915519952774, 0.03741145879030228, ...]"
1,"First, we ob-\nserve that both highly accurate and overly approximate retrieval methods de-\ngrade system efficiency: exact search incurs significant retrieval overhead, while\ncoarse retrieval requires additional reasoning steps during generation.","[0.011610003188252449, 0.026031143963336945, 0.020183976739645004, -0.02932017669081688, 0.009143228642642498, 0.020352644845843315, 0.008602084591984749, -0.011982479132711887, -0.03252487629652023, 0.04742391034960747, 0.0696600154042244, -0.08349081873893738, -0.026790151372551918, -0.0133247971534729, 0.013099906034767628, 0.005457122810184956, -0.013598883524537086, -0.062351055443286896, -0.0026143589057028294, -0.012572817504405975, 0.027802161872386932, -0.02612953446805477, 0.007077041547745466, -0.025988977402448654, 0.029432622715830803, -0.02307944931089878, 0.05237151309847832, 0.055210765451192856, 0.04050850868225098, -0.041070736944675446, -0.0035719031002372503, -0.03319954872131348, 0.0007445125374943018, -0.011153193190693855, 0.00475082453340292, 0.028743892908096313, 0.033255770802497864, -0.011919228360056877, 0.000934703

In [26]:
semantic_embeddings = GenerateEmbeddings(
    client=azure_client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    chunk_table_name="`24280021-pa3`.processed_data.semantic_chunks"
)

table_name = "`24280021-pa3`.processed_data.semantic_embeddings"
semantic_embeddings.embed()
semantic_embeddings.save_to_table(table_name)

Generating embeddings for 341 chunks... This might take a minute.
Embeddings generated successfully!
Saved to `24280021-pa3`.processed_data.semantic_embeddings


### Task 4: Simulating Vector Search

In [ ]:
class VectorSearch:
    def __init__(self, client, model_name, embeddings_table):
        self.client = client
        self.model_name = model_name
        self.embeddings_table = embeddings_table

    def _get_embedding(self, text):
        response = self.client.embeddings.create(
            model=self.model_name,
            input=text
        )
        return response.data[0].embedding
    

    def search_vectors(self, query, top_k=3):
        ### 1. Embed the query
        query_embedding = self._get_embedding(query)
        embeddings_df = spark.read.table(self.embeddings_table)
        
        # Nested function
        def calculate_similarity(emb2):
            vec1 = np.array(query_embedding)
            vec2 = np.array(emb2)
            
            norm1 = np.linalg.norm(vec1)
            norm2 = np.linalg.norm(vec2)
            
            if norm1 == 0 or norm2 == 0:
                return 0.0
            return float(np.dot(vec1, vec2) / (norm1 * norm2))
            
        ### 2. Calculate the cosine similarity between the query and each embedding
        cosine_similarity_udf = udf(calculate_similarity, FloatType())
        
        ### 3. Sort the embeddings by similarity and return the top_k
        results = (embeddings_df
            .withColumn("Similarity", cosine_similarity_udf(col("Embedding")))
            .withColumnRenamed("ChunkText", "Chunk_text")
            .orderBy(col("Similarity").desc())
            .limit(top_k)
            .select("Similarity", "Chunk_text")
        )
        
        return results.collect()

In [9]:
azure_client = AzureOpenAI(
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT")
)

In [12]:
vs_fixed = VectorSearch(
    client = azure_client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table = "`24280021-pa3`.processed_data.fixed_embeddings"
)
results_1 = vs_fixed.search_vectors("What is SearchAgent-X?")
display(results_1)


[Row(Similarity=0.64822918176651, Chunk_text="X\n\n3.1 Overall Architecture\n\nDrawing upon the above insights, we propose SearchAgent-X, an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. Figure 3 shows SearchAgent-X's architecture, a tightly integrated system processing search agent requests at the token generation level. At each LLM output step, the system checks for special tags that trigger the Retriever for an ANN-based search (e.g., <search>) or "),
 Row(Similarity=0.6472952365875244, Chunk_text='Agent-X Execution\n\nThis section outlines the high-level execution flow of the SearchAgent-X system, as depicted in Algorithm 1, complementing the conceptual component descriptions in Section 3 of the main paper.\nSearchAgent-X orchestrates LLM inference (with prefix caching, Section 3.1), dynamic high-recall approximate retrieval, Priority Scheduling (Section 3.2), and Non-Stall Re

In [13]:
vs_semantic = VectorSearch(
    client = azure_client,
    model_name=os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    embeddings_table = "`24280021-pa3`.processed_data.semantic_embeddings"
)
results_2 = vs_semantic.search_vectors("What is SearchAgent-X?")
display(results_2)

[Row(Similarity=0.6967448592185974, Chunk_text='SearchAgent-X is designed to optimize end-to-end system throughput and latency by smoothly coordinating the interleaving of self-reasoning and retrieval. Since both overly low and high retrieval efforts lead to degraded efficiency, SearchAgent-X chooses to build upon a high-recall approximate retrieval method. To tackle the problem of improper scheduling, SearchAgent-X schedules requests with priority awareness through their real-time status to enhance KV-cache utilization. Moreover, in order to overcome frequent retrieval stalls, SearchAgent-X proposes a non-stall retrieval mechanism through an adaptive mechanism that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality.'),
 Row(Similarity=0.6721732020378113, Chunk_text='SearchAgent-X is available at\nhttps://github.com/tiannuo-yang/SearchAgent-X. 1 Introduction\n\nTraditional Retrieval-Augmented Generation (RAG) typically uses a sequential

### Task 5: RAG Architecture

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"), 
    temperature=0.0
)

system_message = SystemMessagePromptTemplate.from_template(
    """ 
    `Role:`
    You are a strictly bounded technical assistant. 
    
    `Task:`
    You must answer the user's question using ONLY the provided context below. 
    
    `Constraint:`
    If the provided context does not contain the answer, you must strictly reply with exactly: 'I do not know'. Do not attempt to guess or use outside knowledge.\n\n"
    
    `Context:`\n{context}"
    """
)

human_message = HumanMessagePromptTemplate.from_template(
    """ 
    `Question:`
    {question}
    """
)

chat_prompt = ChatPromptTemplate.from_messages(
    [system_message, human_message]
)

class RAGPipeline:
    def __init__(self, llm, chat_prompt, vector_search):
        self.llm=llm 
        self.chat_prompt=chat_prompt
        self.vector_search=vector_search
    
    def get_context(self, question):
        contexts = self.vector_search.search_vectors(question, top_k=3)
        context = ""
        context = "\n\n".join([row.Chunk_text for row in contexts])
        return context

    def invoke(self, question):
        content = None
        retrieved_context = self.get_context(question)
        prompt = self.chat_prompt.format_messages(
                    context=retrieved_context, 
                    question=question
                )
        response = self.llm.invoke(prompt)
        content = response.content
        return content

    def invoke_w_lcel(self, question):
        chain = (
            {
                "question": RunnablePassthrough(),              # RunnablePassthrough() grabs the raw 'question' string passed into chain.invoke()
                "context": lambda q: self.get_context(q)        # input (q) passed to the function
            }
            | self.chat_prompt 
            | self.llm 
            | StrOutputParser()
        )
        return chain.invoke(question)


In [20]:
rag_fixed = RAGPipeline(llm, chat_prompt, vs_fixed)
query = "What is SearchAgent-X?"
answer = rag_fixed.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

answer = rag_fixed.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It is a tightly integrated system that processes search agent requests at the token generation level, orchestrating LLM inference with prefix caching, dynamic high-recall approximate retrieval, Priority Scheduling, and Non-Stall Retrieval to achieve efficient search agents.

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end efficiency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It is a tightly integrated system that processes search agent requests at the token generation level, orchestrating LLM inference with prefix caching, dynamic high-recall approximate retrieval, Priority Scheduling, and Non-Stall Retrieval to achieve efficient search agents.


In [21]:

rag_semantic = RAGPipeline(llm, chat_prompt, vs_semantic)
query = "What is SearchAgent-X?"
answer = rag_semantic.invoke(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

print()

answer = rag_semantic.invoke_w_lcel(query)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end system throughput and latency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It builds upon a high-recall approximate retrieval method and schedules requests with priority awareness through their real-time status to enhance KV-cache utilization. Additionally, it employs a non-stall retrieval mechanism that allows generation to proceed without unnecessary waiting while ensuring sufficient retrieval quality. SearchAgent-X processes search agent requests at the token generation level and incorporates a priority scheduler to optimize GPU resource usage.

Question: What is SearchAgent-X?
Answer: SearchAgent-X is an inference system explicitly designed to optimize end-to-end system throughput and latency for search agent workloads by smoothly interleaving self-reasoning and retrieval. It builds upon a high-recall approximate retrieval met

## Part 2: Databricks Managed Vector Search and Agentic RAG Flow

### Task 1: Data Preparation

In [0]:
import os
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from databricks.vector_search.client import VectorSearchClient
from dotenv import load_dotenv

load_dotenv()

True

In [0]:
### TO-DO: Prepare the fixed chunks table

### Task 2: Vector Search Management

In [0]:
# FUNCTIONS I MADE IN CASE YOU GUYS NEED TO CLEANUP YOUR WORKSPACE

def vector_store_endpoint_cleanup(vsc):
    endpoints = vsc.list_endpoints().get('endpoints', [])

    if not endpoints:
        print("No endpoints found. Workspace is clear.")
    else:
        for e in endpoints:
            name = e['name']
            print(f"Deleting endpoint: {name}...")
            vsc.delete_endpoint(name)
        
        print("Waiting for workspace quota to refresh...")
        while len(vsc.list_endpoints().get('endpoints', [])) > 0:
            print("Endpoint still terminating... waiting 5s", end="\r")
            time.sleep(5)
        print("\nWorkspace cleared. Quota reset to 0/1.")

def vector_store_index_cleanup(vsc, endpoint_name, index_full_name):
    """Delete the vector store endpoint if it exists"""
    try:
        print(f"Attempting to delete index: {index_full_name}...")
        vsc.delete_index(endpoint_name=endpoint_name, index_name=index_full_name)
        print("Index deletion command sent.")
    except Exception as e:
        print(f"Index not found or already deleted: {e}")

    try:
        print(f"Deleting endpoint: {endpoint_name}...")
        vsc.delete_endpoint(name=endpoint_name)
    except Exception as e:
        print(f"Endpoint not found or already deleted: {e}")

    print("Waiting for Unity Catalog to clear metadata...waiting 5s")
    time.sleep(5)
    print("System cleared. You can now run the creation script.")


In [0]:
vsc = VectorSearchClient()
endpoint_name = ...
index_name = ...

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
# vector_store_endpoint_cleanup(vsc)
# vector_store_index_cleanup(vsc, endpoint_name, index_name)

In [0]:
print(f"Creating Endpoint: {endpoint_name}...")
### TO-DO: Create the vector store endpoint

print(f"\nCreating index: {index_name}...")
### TO-DO: Create the vector store index

print("Index creation initiated.")


### Task 3: Adaptive RAG Workflow

In [0]:
llm = AzureChatOpenAI(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"), 
    api_version=os.environ.get("OPENAI_API_VERSION"), 
    temperature=0.0
)

class GraphState(TypedDict):
    question: str
    initial_answer: str
    grade: str
    final_answer: str
    context: str

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    ### TO-DO: Retrieve the top-k most relevant chunks from the vector store
    results = VectorSearchClient().get_index().similarity_search()
    return ""

### TO-DO
context_retriever_node = ...

### TO-DO
system_message = SystemMessagePromptTemplate.from_template(
)

### TO-DO
human_message = HumanMessagePromptTemplate.from_template(
    
)

### TO-DO
chat_prompt = ChatPromptTemplate.from_messages()

### TO-DO
rag_chain = (

)

def base_rag_node(state: GraphState) -> dict:
    ### TO-DO
    return {}




In [0]:
### TO-DO
class AnswerGrader(BaseModel):
    binary_score: str = Field()

### TO-DO
grader_system_message = SystemMessagePromptTemplate.from_template(
)

### TO-DO
grader_human_message = HumanMessagePromptTemplate.from_template(
)

### TO-DO
grader_prompt = ChatPromptTemplate.from_messages()

### TO-DO
grader_chain = ...


def grader_node(state: GraphState) -> dict:
    ### TO-DO
    return {}



In [0]:
### TO-DO
class CategoryRouter(BaseModel):
    category: str = Field()

### TO-DO
router_prompt = ChatPromptTemplate.from_template(
)

### TO-DO
router_chain = ...


def text_file_fallback_node(state: GraphState) -> dict:
    ### TO-DO
    category = ...
    
    file_map = {
        "databricks": "/path/to/databricks_info.txt",
        "azure": "/path/to/azure_info.txt",
    }
    
    file_path = file_map.get(category, None)
    if file_path is None:
        return {"final_answer": f"Category not found: {category}"}
    
    try:
        with open(file_path, "r") as f:
            full_text = f.read()
            
        response = llm.invoke(
            ### TO-DO
        )
        return {}
        
    except Exception as e:
        return {"final_answer": f"Fallback failed. Could not read file at {file_path}. Error: {e}"}


### Task 4: Creating the Agentic RAG Graph

In [0]:

### TO-DO: Using these functions, create a state graph that will answer, grade, reanswer a questions
workflow = StateGraph()
workflow.add_node()
workflow.add_edge()

workflow.add_conditional_edges(
)
compiled_adaptive_rag = workflow.compile()


### Task 5: Executing tbe Agentic RAG FLow

In [0]:

inputs = {"question": "What is Databricks?"}
final_state = None

print(f"--- STARTING ADAPTIVE RAG FLOW ---")
print(f"User Question: {inputs['question']}\n")

### TO-DO


--- STARTING ADAPTIVE RAG FLOW ---
User Question: What is Databricks?

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[local_rag_node] {'initial_answer': 'The provided context does not contain information about what Databricks is. Therefore, I do not know the answer based on the given context.'}
[grader_node] {'grade': 'no'}
[file_fallback] {'final_answer': 'Databricks is a unified data platform known as the pioneer of the Lakehouse Architecture, which combines the best elements of data lakes and data warehouses. Built on top of Apache Spark, Delta Lake, and MLflow, it provides a collaborative environment for data engin

## MCP

In [0]:
%pip install --upgrade mlflow[databricks] databricks-agents
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import time
import yaml
import mlflow
from dotenv import load_dotenv
from mlflow.pyfunc import ResponsesAgent
from typing import Any, Dict
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from databricks.vector_search.client import VectorSearchClient

load_dotenv()


True

In [0]:
def vector_search_tool(query: str) -> str:
    ### TO-DO: Implement a function that calls the vector store endpoint and write a detailed docstring for the function.
    try:
        return ""
    except Exception as e:
        return f"Database search failed: {e}"

def vector_store_endpoint_cleanup(vsc):
    ### TO-DO: Implement a function that deletes the vector store endpoint if they exist and write a detailed docstring for the function
    pass



def vector_store_index_cleanup(vsc, endpoint_name, index_full_name):
    ### TO-DO: Implement a function that deletes the vector store index if they exist and write a detailed docstring for the function
    pass


In [0]:

class VectorSearchAgent(ResponsesAgent):
    def __init__(self, config_path: str):
        with open(config_path, 'r') as f:
            self.config = yaml.safe_load(f)
        super().__init__()
        
        self.system_prompt = (
            "You are a precise data engineering assistant. "
            "To answer questions about the course or documentation, you MUST use "
            "the provided vector_search_tool to retrieve context. "
            "If the tool does not return relevant information, state that you do not know."
        )

    def predict(self, payload) -> Dict[str, Any]:
        """Executes the Agent Loop locally."""
        if isinstance(payload, dict):
            messages = payload.get("messages", payload.get("input", []))
        elif isinstance(payload, list):
            messages = payload
        else:
            messages = getattr(payload, "messages", getattr(payload, "input", []))
            
        user_question = messages[-1]['content'] if isinstance(messages[-1], dict) else messages[-1].content
        
        ### TO-DO
        llm = AzureChatOpenAI(
        )
        
        prompt_template = ChatPromptTemplate.from_messages([
           
        ])
        
        tools = ...
        agent_logic = ...
        agent_executor = ...
        
        # RUN THE LOOP
        raw_response = agent_executor.invoke({"input": user_question})["output"]
        
        return {"choices": [{"message": {"role": "assistant", "content": raw_response}}]}

agent = VectorSearchAgent(config_path="agent_config.yml")
mlflow.models.set_model(model=agent)


🚀 Starting Local Agent Executor Test...

User Query: What is the Databricks Vector Search endpoint cleanup function supposed to do?


> Entering new AgentExecutor chain...

Invoking: `vector_search_tool` with `{'query': 'Databricks Vector Search endpoint cleanup function'}`


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
 yet yields the significant benefit of better cache utilization. More ablation results can be found in Appendix C.2 and C.3.
5 Related Work
Prior work on end-to-end RAG efficiency—spanning caching [18, 29, 30], pipelining [31, 32], and hyperparameter tuning [17, 33]—primarily targets the traditional se

[Trace(trace_id=tr-ff67891e25ae987f42225e7355827b15), Trace(trace_id=tr-1ec06505de84197ad18e3b23155432a5)]

### Task 4: Executing and Testing

In [0]:
test_questions = [
    "What is the Databricks Vector Search endpoint cleanup function supposed to do?",
    "What tools do you have and what do they do?"
]

print("Starting Local Agent Executor Test...\n" + "="*50)

for question in test_questions:
    print(f"\nUser Query: {question}")
    
    # Required schema structure
    payload = {
        "input": [{"role": "user", "content": question}], 
        "messages": [{"role": "user", "content": question}]
    }
    
    # Invoke the agent
    response = agent.predict(payload)
    final_answer = response["choices"][0]["message"]["content"]
    
    print(f"Final Answer:\n{final_answer}")
    print("-" * 50)